In [16]:
import pandas as pd
from pathlib import Path
import gc

from schema import SCHEMA_CURICA

In [17]:
# toma como diretório pai o diretório do arquivo do notebook
BASE_DIR = Path("etl_censo.ipynb").resolve().parent
RAW_CENSO_DIR = BASE_DIR / "data" / "raw"
RAW_ANEXOS = RAW_CENSO_DIR / "anexos"

In [18]:
df_escola_2025 = pd.read_csv(RAW_CENSO_DIR / "Tabela_Escola_2025.csv", sep=";", encoding="latin-1", low_memory=False)
df_matricula_2025 = pd.read_csv(RAW_CENSO_DIR / "Tabela_Matricula_2025.csv", sep=";", encoding="latin-1", low_memory=False)
df_turma_2025 = pd.read_csv(RAW_CENSO_DIR / "Tabela_Turma_2025.csv", sep=";", encoding="latin-1", low_memory=False)
df_docente_2025 = pd.read_csv(RAW_CENSO_DIR / "Tabela_Docente_2025.csv", sep=";", encoding="latin-1", low_memory=False)

df_escola_ac_2025 = df_escola_2025[df_escola_2025["SG_UF"] == "AC"].copy()
df_matricula_ac_2025 = df_matricula_2025[df_matricula_2025["CO_ENTIDADE"].isin(df_escola_ac_2025["CO_ENTIDADE"])].copy()
df_turma_ac_2025 = df_turma_2025[df_turma_2025["CO_ENTIDADE"].isin(df_escola_ac_2025["CO_ENTIDADE"])].copy()
df_docente_ac_2025 =  df_docente_2025[df_docente_2025["CO_ENTIDADE"].isin(df_escola_ac_2025["CO_ENTIDADE"])].copy()

del df_escola_2025
del df_matricula_2025
del df_turma_2025
del df_docente_2025

gc.collect()

24

In [19]:
df_escola_ac_2025 = df_escola_ac_2025[df_escola_ac_2025["TP_SITUACAO_FUNCIONAMENTO"] == 1].copy()

In [20]:
df_censo_ac_2025 = df_escola_ac_2025.merge(
    df_matricula_ac_2025,
    on="CO_ENTIDADE",
    how="left",
    suffixes=("", "_mat")
)

df_censo_ac_2025 = df_censo_ac_2025.merge(
    df_turma_ac_2025,
    on="CO_ENTIDADE",
    how="left",
    suffixes=("", "_tur")
)

df_censo_ac_2025 = df_censo_ac_2025.merge(
    df_docente_ac_2025,
    on="CO_ENTIDADE",
    how="left",
    suffixes=("", "_doc")
)

In [21]:
del df_escola_ac_2025
del df_matricula_ac_2025
del df_turma_ac_2025
del df_docente_ac_2025

gc.collect()

0

In [22]:
df_censo_ac_2025 = df_censo_ac_2025[df_censo_ac_2025['TP_DEPENDENCIA'] == 2]

df_censo_ac_2025 = df_censo_ac_2025[SCHEMA_CURICA]

df_censo_ac_2025.shape

(592, 316)

In [23]:
df_anexos = pd.read_csv(RAW_ANEXOS / "anexos_escolas_estaduais.csv", sep=";", encoding="latin-1", low_memory=False)



In [24]:
df_anexos = df_anexos.rename(columns={'Código da Escola': 'CO_ENTIDADE'})



In [25]:
df_anexos.columns

Index([' MUNICÍPIO ', 'CO_ENTIDADE', 'ESCOLA ', 'Localização',
       'MODALIDADE DE ENSINO', 'ENDEREÇOS ', 'ATIVA(1)'],
      dtype='str')

In [26]:
df_anexos['CO_ENTIDADE'].nunique()

565

In [27]:
codigos_com_anexos = df_anexos['CO_ENTIDADE'].value_counts()
codigos_com_anexos = codigos_com_anexos[codigos_com_anexos > 1].index

df_censo_ac_2025['tem_anexo'] = df_censo_ac_2025['CO_ENTIDADE'].isin(codigos_com_anexos).astype(int)

df_censo_ac_2025['tem_anexo'].value_counts()

tem_anexo
0    519
1     73
Name: count, dtype: int64

## Teste Modelos com colunas selecionadas

In [15]:
# ==========================================
# 1. IMPORTS
# ==========================================
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report




# ==========================================
# 2. DEFINIÇÃO DAS FEATURES
# ==========================================

cat_cols = ['TP_LOCALIZACAO', 'IN_LOCAL_FUNC_PREDIO_ESCOLAR','TP_OCUPACAO_PREDIO_ESCOLAR',
            'IN_LOCAL_FUNC_GALPAO', 'IN_LOCAL_FUNC_SALAS_OUTRA_ESC', 'IN_LOCAL_FUNC_OUTROS',
            'IN_PREDIO_COMPARTILHADO'
]

num_cols = ['QT_SALAS_UTILIZADAS_DENTRO', 'QT_SALAS_UTILIZADAS_FORA', 'QT_SALAS_UTILIZADAS',
            'QT_MAT_BAS', 'QT_MAT_INF', 'QT_MAT_FUND', 'QT_MAT_FUND_AI', 'QT_MAT_FUND_AF',
            'QT_MAT_MED', 'QT_MAT_MED_PROP_NS', 'QT_MAT_BAS_INT', 'QT_MAT_INF_CRE_D', 'QT_MAT_INF_CRE_DM',
            'QT_MAT_INF_CRE_DV', 'QT_MAT_INF_PRE_D', 'QT_MAT_INF_PRE_DM', 'QT_MAT_INF_PRE_DV',
            'QT_MAT_FUND_D', 'QT_MAT_FUND_DM', 'QT_MAT_FUND_DV', 'QT_MAT_MED_D', 'QT_MAT_MED_DM',
            'QT_MAT_MED_DV', 'QT_DOC_BAS', 'QT_TUR_BAS', 'QT_TUR_INF', 'QT_TUR_INF_CRE',
            'QT_TUR_INF_PRE', 'QT_TUR_FUND', 'QT_TUR_FUND_AI', 'QT_TUR_FUND_AI_1', 'QT_TUR_FUND_AI_2',
            'QT_TUR_FUND_AI_3', 'QT_TUR_FUND_AI_4', 'QT_TUR_FUND_AI_5', 'QT_TUR_FUND_AI_MULTIETAPA',
            'QT_TUR_FUND_AF', 'QT_TUR_FUND_AF_6', 'QT_TUR_FUND_AF_7', 'QT_TUR_FUND_AF_8',
            'QT_TUR_FUND_AF_9', 'QT_TUR_FUND_AF_MULTI', 'QT_TUR_FUND_AF_CORRFLUXO', 'QT_TUR_MED',
            'QT_TUR_MED_PROP', 'QT_TUR_MED_PROP_1', 'QT_TUR_MED_PROP_2', 'QT_TUR_MED_PROP_3', 'QT_TUR_MED_PROP_4',
            'QT_TUR_MED_PROP_NS', 'QT_TUR_BAS_D', 'QT_TUR_BAS_DM', 'QT_TUR_BAS_DV', 'QT_TUR_BAS_N',
            'QT_TUR_INF_CRE_D',  'QT_TUR_INF_CRE_DM', 'QT_TUR_INF_CRE_DV', 'QT_TUR_INF_CRE_N',
            'QT_TUR_INF_PRE_D', 'QT_TUR_INF_PRE_DM',  'QT_TUR_INF_PRE_DV', 'QT_TUR_INF_PRE_N', 'QT_TUR_FUND_D',
            'QT_TUR_FUND_DM', 'QT_TUR_FUND_DV', 'QT_TUR_FUND_N', 'QT_TUR_FUND_AI_D', 'QT_TUR_FUND_AI_DM',
            'QT_TUR_FUND_AI_DV', 'QT_TUR_FUND_AI_N', 'QT_TUR_FUND_AF_D', 'QT_TUR_FUND_AF_DM',
            'QT_TUR_FUND_AF_DV', 'QT_TUR_FUND_AF_N', 'QT_TUR_MED_D', 'QT_TUR_MED_DM', 'QT_TUR_MED_DV', 'QT_TUR_MED_N',
            'QT_DOC_BAS_VINCULO_CONCUR', 'QT_DOC_BAS_VINCULO_CONTRA', 'QT_DOC_BAS_VINCULO_TERCEIR', 'QT_DOC_BAS_VINCULO_CLT'
]


# ==========================================
# 3. MONTAGEM DO DATASET
# ==========================================

X = df_censo_ac_2025[cat_cols + num_cols].copy()
y = df_censo_ac_2025['tem_anexo'].copy()

# Garantia de integridade
X = X.loc[:, ~X.columns.duplicated()]


# ==========================================
# 4. SPLIT (ANTES DE QUALQUER FIT)
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


# ==========================================
# 5. PIPELINES DE TRANSFORMAÇÃO
# ==========================================

# CATEGÓRICAS
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('to_str', FunctionTransformer(lambda x: x.astype(str))),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# NUMÉRICAS
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


# ==========================================
# 6. PREPROCESSADOR
# ==========================================

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', cat_pipeline, cat_cols),
        ('num', num_pipeline, num_cols)
    ]
)


# ==========================================
# 7. MODELO
# ==========================================

# modelo = LogisticRegression(
#     max_iter=1000,
#     class_weight='balanced',
#     random_state=42
# )

modelo = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)


# ==========================================
# 8. PIPELINE FINAL
# ==========================================

pipeline = Pipeline(steps=[
    ('preprocessamento', preprocessador),
    ('modelo', modelo)
])


# ==========================================
# 9. TREINO
# ==========================================

pipeline.fit(X_train, y_train)


# ==========================================
# 10. AVALIAÇÃO
# ==========================================

y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.99      0.96       156
           1       0.86      0.55      0.67        22

    accuracy                           0.93       178
   macro avg       0.90      0.77      0.81       178
weighted avg       0.93      0.93      0.93       178



## Teste para avalição de relevância da coluna

In [49]:
# ==========================================
# 1. IMPORTS
# ==========================================
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

from sklearn.inspection import permutation_importance


# ==========================================
# 2. DEFINIÇÃO DAS FEATURES
# ==========================================

cat_cols = [
    'TP_LOCALIZACAO', 'IN_LOCAL_FUNC_PREDIO_ESCOLAR','TP_OCUPACAO_PREDIO_ESCOLAR',
    'IN_LOCAL_FUNC_GALPAO', 'IN_LOCAL_FUNC_SALAS_OUTRA_ESC', 'IN_LOCAL_FUNC_OUTROS',
    'IN_PREDIO_COMPARTILHADO'
]

num_cols = [
    'QT_SALAS_UTILIZADAS_DENTRO', 'QT_SALAS_UTILIZADAS_FORA', 'QT_SALAS_UTILIZADAS',
    'QT_MAT_BAS', 'QT_MAT_INF', 'QT_MAT_FUND', 'QT_MAT_FUND_AI', 'QT_MAT_FUND_AF',
    'QT_MAT_MED', 'QT_MAT_MED_PROP_NS', 'QT_MAT_BAS_INT', 'QT_MAT_INF_CRE_D', 'QT_MAT_INF_CRE_DM',
    'QT_MAT_INF_CRE_DV', 'QT_MAT_INF_PRE_D', 'QT_MAT_INF_PRE_DM', 'QT_MAT_INF_PRE_DV',
    'QT_MAT_FUND_D', 'QT_MAT_FUND_DM', 'QT_MAT_FUND_DV', 'QT_MAT_MED_D', 'QT_MAT_MED_DM',
    'QT_MAT_MED_DV', 'QT_DOC_BAS', 'QT_TUR_BAS', 'QT_TUR_INF', 'QT_TUR_INF_CRE',
    'QT_TUR_INF_PRE', 'QT_TUR_FUND', 'QT_TUR_FUND_AI', 'QT_TUR_FUND_AI_1', 'QT_TUR_FUND_AI_2',
    'QT_TUR_FUND_AI_3', 'QT_TUR_FUND_AI_4', 'QT_TUR_FUND_AI_5', 'QT_TUR_FUND_AI_MULTIETAPA',
    'QT_TUR_FUND_AF', 'QT_TUR_FUND_AF_6', 'QT_TUR_FUND_AF_7', 'QT_TUR_FUND_AF_8',
    'QT_TUR_FUND_AF_9', 'QT_TUR_FUND_AF_MULTI', 'QT_TUR_FUND_AF_CORRFLUXO', 'QT_TUR_MED',
    'QT_TUR_MED_PROP', 'QT_TUR_MED_PROP_1', 'QT_TUR_MED_PROP_2', 'QT_TUR_MED_PROP_3', 'QT_TUR_MED_PROP_4',
    'QT_TUR_MED_PROP_NS', 'QT_TUR_BAS_D', 'QT_TUR_BAS_DM', 'QT_TUR_BAS_DV', 'QT_TUR_BAS_N',
    'QT_TUR_INF_CRE_D', 'QT_TUR_INF_CRE_DM', 'QT_TUR_INF_CRE_DV', 'QT_TUR_INF_CRE_N',
    'QT_TUR_INF_PRE_D', 'QT_TUR_INF_PRE_DM', 'QT_TUR_INF_PRE_DV', 'QT_TUR_INF_PRE_N',
    'QT_TUR_FUND_D', 'QT_TUR_FUND_DM', 'QT_TUR_FUND_DV', 'QT_TUR_FUND_N',
    'QT_TUR_FUND_AI_D', 'QT_TUR_FUND_AI_DM', 'QT_TUR_FUND_AI_DV', 'QT_TUR_FUND_AI_N',
    'QT_TUR_FUND_AF_D', 'QT_TUR_FUND_AF_DM', 'QT_TUR_FUND_AF_DV', 'QT_TUR_FUND_AF_N',
    'QT_TUR_MED_D', 'QT_TUR_MED_DM', 'QT_TUR_MED_DV', 'QT_TUR_MED_N',
    'QT_DOC_BAS_VINCULO_CONCUR', 'QT_DOC_BAS_VINCULO_CONTRA',
    'QT_DOC_BAS_VINCULO_TERCEIR', 'QT_DOC_BAS_VINCULO_CLT'
]


# ==========================================
# 3. DATASET
# ==========================================

X = df_censo_ac_2025[cat_cols + num_cols].copy()
y = df_censo_ac_2025['tem_anexo'].copy()

X = X.loc[:, ~X.columns.duplicated()]


# ==========================================
# 4. SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


# ==========================================
# 5. PIPELINES
# ==========================================

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('to_str', FunctionTransformer(lambda x: x.astype(str))),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


# ==========================================
# 6. PREPROCESSADOR
# ==========================================

preprocessador = ColumnTransformer([
    ('cat', cat_pipeline, cat_cols),
    ('num', num_pipeline, num_cols)
])


# ==========================================
# 7. MODELO
# ==========================================

modelo = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

#modelo = RandomForestClassifier(
#    n_estimators=200,
#    random_state=42,
#    n_jobs=-1,
#    class_weight='balanced'
#)


# ==========================================
# 8. PIPELINE FINAL
# ==========================================

pipeline = Pipeline([
    ('preprocessamento', preprocessador),
    ('modelo', modelo)
])


# ==========================================
# 9. TREINO
# ==========================================

pipeline.fit(X_train, y_train)


# ==========================================
# 10. AVALIAÇÃO
# ==========================================

y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))


# ==========================================
# 11. PERMUTATION IMPORTANCE
# ==========================================

print("\nCalculando permutation importance...")

result = permutation_importance(
    pipeline,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

# IMPORTANTE: aqui usamos X_test ORIGINAL (antes do one-hot)
df_importancia = pd.DataFrame({
    'feature': X_test.columns,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

print("\nTop 20 variáveis mais importantes:")
print(df_importancia.head(20))

              precision    recall  f1-score   support

           0       0.97      0.93      0.95       156
           1       0.62      0.82      0.71        22

    accuracy                           0.92       178
   macro avg       0.80      0.87      0.83       178
weighted avg       0.93      0.92      0.92       178


Calculando permutation importance...

Top 20 variáveis mais importantes:
                          feature  importance
4   IN_LOCAL_FUNC_SALAS_OUTRA_ESC    0.076404
71                 QT_TUR_FUND_DV    0.041573
46               QT_TUR_FUND_AF_8    0.041011
79              QT_TUR_FUND_AF_DV    0.038202
29                  QT_MAT_MED_DV    0.038202
0                  TP_LOCALIZACAO    0.023034
16             QT_MAT_MED_PROP_NS    0.021910
47               QT_TUR_FUND_AF_9    0.021348
77               QT_TUR_FUND_AF_D    0.020787
43                 QT_TUR_FUND_AF    0.020787
59                  QT_TUR_BAS_DV    0.017978
54              QT_TUR_MED_PROP_3    0.017416
3

In [50]:
top_features = df_importancia.head(15)['feature'].tolist()

print(top_features)

['IN_LOCAL_FUNC_SALAS_OUTRA_ESC', 'QT_TUR_FUND_DV', 'QT_TUR_FUND_AF_8', 'QT_TUR_FUND_AF_DV', 'QT_MAT_MED_DV', 'TP_LOCALIZACAO', 'QT_MAT_MED_PROP_NS', 'QT_TUR_FUND_AF_9', 'QT_TUR_FUND_AF_D', 'QT_TUR_FUND_AF', 'QT_TUR_BAS_DV', 'QT_TUR_MED_PROP_3', 'IN_LOCAL_FUNC_GALPAO', 'QT_TUR_FUND', 'QT_TUR_FUND_D']


## Teste com colunas relevantes

In [56]:
# ==========================================
# 1. IMPORTS
# ==========================================
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report




# ==========================================
# 2. DEFINIÇÃO DAS FEATURES
# ==========================================

cat_cols = ['IN_LOCAL_FUNC_SALAS_OUTRA_ESC', 'TP_LOCALIZACAO']

num_cols = ['QT_TUR_FUND_DV', 'QT_TUR_FUND_AF_8', 'QT_TUR_FUND_AF_DV', 'QT_MAT_MED_DV', 'QT_MAT_MED_PROP_NS', 'QT_TUR_FUND_AF_9', 'QT_TUR_FUND_AF_D'
]


# ==========================================
# 3. MONTAGEM DO DATASET
# ==========================================

X = df_censo_ac_2025[cat_cols + num_cols].copy()
y = df_censo_ac_2025['tem_anexo'].copy()

# Garantia de integridade
X = X.loc[:, ~X.columns.duplicated()]


# ==========================================
# 4. SPLIT (ANTES DE QUALQUER FIT)
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


# ==========================================
# 5. PIPELINES DE TRANSFORMAÇÃO
# ==========================================

# CATEGÓRICAS
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('to_str', FunctionTransformer(lambda x: x.astype(str))),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# NUMÉRICAS
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


# ==========================================
# 6. PREPROCESSADOR
# ==========================================

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', cat_pipeline, cat_cols),
        ('num', num_pipeline, num_cols)
    ]
)


# ==========================================
# 7. MODELO
# ==========================================

modelo = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

#modelo = RandomForestClassifier(
#    n_estimators=200,
#    random_state=42,
#    n_jobs=-1,
#    class_weight='balanced'
#)


# ==========================================
# 8. PIPELINE FINAL
# ==========================================

pipeline = Pipeline(steps=[
    ('preprocessamento', preprocessador),
    ('modelo', modelo)
])


# ==========================================
# 9. TREINO
# ==========================================

pipeline.fit(X_train, y_train)


# ==========================================
# 10. AVALIAÇÃO
# ==========================================

y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.94      0.96       156
           1       0.68      0.86      0.76        22

    accuracy                           0.93       178
   macro avg       0.83      0.90      0.86       178
weighted avg       0.94      0.93      0.94       178

